# **Construction of Layered Quadripartite Networks (LQNs)**

This notebook also produces the AT_MN family of artifacts 

# Import Libraries and Configurations

In [1]:
import os
import sys

import pandas as pd

# Add project root to Python's path
sys.path.append(os.path.abspath(os.path.join('..')))

from config import (
    NETWORK_DATA_DIRS,
    NETWORK_FILES,
    REACTOME_DATA_DIRS,
)

# Functions

## Create Tripartite Messenger RNA-Centered Network

In [2]:
def build_mrna_network(pathway_edges_df, pathway_nodes_df, mrna_ids):
    """
    Extract an mRNA-centered subnetwork from a flattened pathway network.

    This function constructs a subnetwork containing messenger RNAs of
    interest and their associated pathway locations and biological pathways.
    The directed edges of the original pathway network are reversed to
    represent relationships from messenger RNAs to pathway locations and
    from pathway locations to pathways. The resulting subnetwork is
    represented by two DataFrames containing the selected edges and nodes.

    Parameters
    ----------
    pathway_edges_df : pandas.DataFrame
        DataFrame containing the directed edges of the flattened pathway
        network, with `source` and `target` columns.
    pathway_nodes_df : pandas.DataFrame
        DataFrame containing the nodes of the flattened pathway network,
        with an `id` column and associated node attributes.
    mrna_ids : list or set
        Collection of messenger RNA identifiers used to select the
        proteins of interest from the pathway network.

    Returns
    -------
    mrna_edges_df : pandas.DataFrame
        DataFrame containing the directed edges of the mRNA-centered
        subnetwork, representing mRNA-location and location-pathway
        relationships.

    mrna_nodes_df : pandas.DataFrame
        DataFrame containing the selected messenger RNA, pathway location,
        and pathway nodes, with their associated attributes.

    Notes
    -----
    - The original pathway network is assumed to contain directed
      relationships from pathways to pathway locations and from pathway
      locations to messenger RNAs.
    - Edge directions are reversed to represent relationships from
      messenger RNAs to pathways through intermediate pathway locations.
    - Messenger RNAs of interest are identified by matching `mrna_ids`
      against the `id` column of the pathway node DataFrame.
    - Pathway locations are selected based on their direct relationships
      with the messenger RNAs of interest.
    - Pathways are selected based on their direct relationships with
      the selected pathway locations.
    - The resulting subnetwork contains the induced edges between all
      selected nodes, including relationships shared by multiple
      messenger RNAs.
    - Messenger RNA identifiers must match the protein node identifiers
      used in the flattened pathway network.
    - The input DataFrames remain unchanged.

    Examples
    --------
    >>> mrna_ids = ['BRCA1', 'TP53', 'EGFR']
    >>> edges_df, nodes_df = build_mrna_network(
    ...     pathway_edges_df,
    ...     pathway_nodes_df,
    ...     mrna_ids
    ... )
    >>> edges_df[['source', 'target']].head()
    >>> nodes_df[['id', 'label', 'type']].head()
    """
    # Reverse the direction of the pathway network edges
    pathway_edges_df = pathway_edges_df.rename(
        columns={
            'source': 'target',
            'target': 'source',
        }
    )[['source', 'target']]

    # Get the proteins encoded by the messenger RNAs of interest
    protein_ids = (
        pathway_nodes_df
        .query('id in @mrna_ids')
        ['id'].to_list()
    )

    # Get the pathway locations associated with the selected proteins
    locations_ids = (
        pathway_edges_df
        .query('source in @protein_ids')
        ['target'].to_list()
    )

    # Get the pathways associated with the selected locations
    pathway_ids = (
        pathway_edges_df
        .query('source in @locations_ids')
        ['target'].to_list()
    )

    # Combine the selected node identifiers in a set
    selected_nodes = (
        set(protein_ids)
        | set(locations_ids)
        | set(pathway_ids)
    )

    # Filter the edges connecting the selected nodes
    mrna_edges_df = (
        pathway_edges_df
        .query(
            '(source in @selected_nodes)'
            ' and (target in @selected_nodes)'
        )
        .reset_index(drop=True)
    )

    # Filter the attributes of the selected nodes
    mrna_nodes_df = (
        pathway_nodes_df
        .query('id in @selected_nodes')
        .reset_index(drop=True)
    )

    return mrna_edges_df, mrna_nodes_df

## Combine Networks

In [3]:
def combine_networks(
    ltn_edges_df, ltn_nodes_df, mrna_edges_df, mrna_nodes_df
):
    """
    Combine microRNA-mRNA interactions with mRNA-pathway subnetworks.

    This function integrates a microRNA-mRNA-pathway network with a
    collection of mRNA-location-pathway subnetworks to construct a
    quadripartite representation. Messenger RNAs associated with pathway
    locations are identified and used to select their corresponding
    microRNAs and regulatory interactions from the original network.
    The selected interactions and nodes are combined with the pathway
    subnetworks, removing duplicate entries.

    Parameters
    ----------
    ltn_edges_df : pandas.DataFrame
        DataFrame containing the edges of the original tripartite
        microRNA-mRNA-pathway network, with `source` and `target` columns.
    ltn_nodes_df : pandas.DataFrame
        DataFrame containing the nodes of the original tripartite network,
        with an `id` column and associated node attributes.
    mrna_edges_df : pandas.DataFrame
        DataFrame containing the edges of the mRNA-location-pathway
        subnetworks, with `source` and `target` columns.
    mrna_nodes_df : pandas.DataFrame
        DataFrame containing the nodes of the mRNA-location-pathway
        subnetworks, including messenger RNAs, pathway locations, and
        pathways, with their associated attributes.

    Returns
    -------
    lqn_edges_df : pandas.DataFrame
        DataFrame containing the directed edges of the combined network,
        representing microRNA-mRNA, mRNA-location, and location-pathway
        relationships.

    lqn_nodes_df : pandas.DataFrame
        DataFrame containing the nodes of the combined network,
        including microRNAs, messenger RNAs, pathway locations, and
        pathways, with their associated attributes.

    Notes
    -----
    - Messenger RNAs of interest are identified from the
      mRNA-location-pathway subnetworks using the `type` attribute
      with the value `Messenger RNA`.
    - MicroRNAs associated with the selected messenger RNAs are
      identified through the original tripartite network edges.
    - Only microRNA-mRNA interactions connecting selected nodes are
      retained from the original network.
    - The selected microRNA-mRNA interactions are concatenated with
      the mRNA-location-pathway subnetworks to construct the combined
      network.
    - Duplicate edges are removed based on all column values.
    - Duplicate nodes are removed based on their `id` attribute,
      preserving the first occurrence of each identifier.
    - Direct mRNA-pathway relationships from the original tripartite
      network are not retained.
    - The input DataFrames remain unchanged.

    Examples
    --------
    >>> edges_df, nodes_df = combine_networks(
    ...     ltn_edges_df,
    ...     ltn_nodes_df,
    ...     mrna_edges_df,
    ...     mrna_nodes_df
    ... )
    >>> edges_df[['source', 'target']].head()
    >>> nodes_df[['id', 'label', 'type']].head()
    """
    # Get the messenger RNAs associated with pathway locations
    mrna_ids = (
        mrna_nodes_df
        .query('type == "Messenger RNA"')
        ['id']
        .to_list()
    )

    # Get the microRNAs associated with the selected messenger RNAs
    mir_ids = (
        ltn_edges_df
        .query('target in @mrna_ids')
        ['source']
        .to_list()
    )

    # Combine the selected microRNA and messenger RNA identifiers
    selected_nodes = (
        set(mir_ids) | set(mrna_ids)
    )

    # Filter the microRNA-mRNA interaction edges
    mir_edges_df = (
        ltn_edges_df
        .query(
            '(source in @selected_nodes)'
            ' and (target in @selected_nodes)'
        )
        .reset_index(drop=True)
    )

    # Filter the microRNA and messenger RNA node attributes
    mir_nodes_df = (
        ltn_nodes_df
        .query('id in @selected_nodes')
        .reset_index(drop=True)
    )

    # Concatenate the microRNA-mRNA and pathway subnetwork edges
    lqn_edges_df = (
        pd.concat(
            objs=[mir_edges_df, mrna_edges_df],
            ignore_index=True
        )
        .drop_duplicates(ignore_index=True)
    )

    # Concatenate the selected node attributes
    lqn_nodes_df = (
        pd.concat(
            objs=[mir_nodes_df, mrna_nodes_df],
            ignore_index=True
        )
        .drop_duplicates(subset='id', ignore_index=True)
    )

    return lqn_edges_df, lqn_nodes_df

In [4]:
def build_quadripartite_network(group):
    """
    Build a quadripartite microRNA-mRNA-location-pathway network.

    This function integrates a tripartite microRNA-mRNA-pathway network
    with flattened Reactome pathway networks to construct a quadripartite
    representation. For each pathway, messenger RNAs of interest are
    identified and used to extract the corresponding mRNA-location-pathway
    subnetwork. These subnetworks are combined with the original
    tripartite network to incorporate semantic pathway locations.
    Pathway enrichment q-values are recovered from the original network.
    The resulting network is represented by two DataFrames containing
    edges and nodes, which are stored in processed CSV files.

    Parameters
    ----------
    group : str
        The name of the group for which the quadripartite network will
        be constructed. The normalized group name must correspond to
        a key in `NETWORK_DATA_DIRS['processed']`.

    Returns
    -------
    lqn_edges_df : pandas.DataFrame
        DataFrame containing the directed edges of the quadripartite
        network, representing microRNA-mRNA, mRNA-location, and
        location-pathway relationships.

    lqn_nodes_df : pandas.DataFrame
        DataFrame containing the nodes of the quadripartite network,
        including microRNAs, messenger RNAs, pathway locations, and
        pathways, with their associated attributes. Pathway nodes
        retain their original enrichment q-values.

    Notes
    -----
    - The following global configuration variables are used:
        * `NETWORK_DATA_DIRS` : Directory paths for processed network data.
        * `NETWORK_FILES` : Filenames for specific network data files.
        * `REACTOME_DATA_DIRS` : Directory paths for processed Reactome
          pathway network data.
    - The group name is normalized by converting it to lowercase and
      replacing spaces with hyphens.
    - Reactome pathway identifiers are normalized by removing the
      `REAC:` prefix from pathway targets and node identifiers.
    - The pathways of interest are identified from the Pathway nodes
      of the original tripartite network.
    - For each pathway, the associated messenger RNAs are identified
      from the original network edges.
    - The function uses `build_mrna_network()` to extract the
      mRNA-location-pathway subnetwork from each flattened pathway network.
    - The extracted subnetworks are collected in lists and concatenated
      after processing all pathways.
    - Empty subnetworks are excluded from concatenation.
    - The function uses `combine_networks()` to integrate the original
      tripartite network with the extracted pathway subnetworks.
    - Pathway enrichment q-values are recovered from the original
      tripartite network using pathway identifiers.
    - The resulting network contains microRNA-mRNA, mRNA-location,
      and location-pathway relationships, excluding direct
      mRNA-pathway edges.
    - The function overwrites the corresponding quadripartite edge
      and node CSV files in the processed directory.

    Examples
    --------
    >>> edges_df, nodes_df = build_quadripartite_network('HER2-enriched')
    >>> edges_df[['source', 'target']].head()
    >>> nodes_df[['id', 'label', 'type', 'qvalue']].head()
    """
    # Define the path to the directory related to the group
    group_dir = (group.lower()).replace(' ', '-')
    group_dir_path = NETWORK_DATA_DIRS['processed'][group_dir]

    # Create a DataFrame for the edges of the tripartite network
    file_name = NETWORK_FILES['tripartite-edges']
    ltn_edges_df = pd.read_csv(os.path.join(group_dir_path, file_name))

    # Create a DataFrame for the nodes of the tripartite network
    file_name = NETWORK_FILES['tripartite-nodes']
    ltn_nodes_df = pd.read_csv(os.path.join(group_dir_path, file_name))

    # Remove the `REAC:` prefix from pathway identifiers
    ltn_edges_df['target'] = \
        ltn_edges_df['target'].str.removeprefix('REAC:')
    ltn_nodes_df['id'] = \
        ltn_nodes_df['id'].str.removeprefix('REAC:')

    # Get the pathways contained in the tripartite network
    pathway_ids = (
        ltn_nodes_df
        .query('type == "Pathway"')
        ['id']
        .to_list()
    )

    # Get the pathway enrichment q-values
    pathway_qvalues_df = (
        ltn_nodes_df
        .query('type == "Pathway"')
        [['id', 'qvalue']]
        .drop_duplicates(subset='id')
    )

    # Create lists to store the group pathway subnetworks
    group_edges = []
    group_nodes = []

    # Define the directory path for flattened pathway networks
    pathway_dir_path = os.path.join(
        REACTOME_DATA_DIRS['processed'], 'flattened-networks'
    )

    # Iterate over the pathways in the network
    for pathway_id in pathway_ids:
        # Get the messenger RNAs (proteins) associated with the pathway
        expressed_mrnas = (
            ltn_edges_df
            .query('target == @pathway_id')
            ['source']
            .to_list()
        )

        # Create a DataFrame for the edges of the flattened pathway
        file_name = pathway_id + NETWORK_FILES['pathway-edges']
        pathway_edges_df = \
            pd.read_csv(os.path.join(pathway_dir_path, file_name))

        # Create a DataFrame for the nodes of the flattened pathway
        file_name = pathway_id + NETWORK_FILES['pathway-nodes']
        pathway_nodes_df = \
            pd.read_csv(os.path.join(pathway_dir_path, file_name))

        # Build the mRNA-location-pathway subnetwork
        mrna_edges_df, mrna_nodes_df = build_mrna_network(
            pathway_edges_df, pathway_nodes_df, expressed_mrnas
        )

        # Store the pathway edges
        if not mrna_edges_df.empty:
            group_edges.append(mrna_edges_df)

        # Store the pathway nodes
        if not mrna_nodes_df.empty:
            group_nodes.append(mrna_nodes_df)

    # Concatenate the pathway edge lists
    group_edges_df = (
        pd.concat(group_edges, ignore_index=True)
        if group_edges
        else pd.DataFrame(columns=ltn_edges_df.columns)
    )

    # Concatenate the pathway node attributes
    group_nodes_df = (
        pd.concat(group_nodes, ignore_index=True)
        if group_nodes
        else pd.DataFrame(columns=ltn_nodes_df.columns)
    )

    # Combine the tripartite network with the pathway subnetworks
    lqn_edges_df, lqn_nodes_df = combine_networks(
        ltn_edges_df,
        ltn_nodes_df,
        group_edges_df,
        group_nodes_df
    )

    # Recover the pathway enrichment q-values
    lqn_nodes_df = (
        lqn_nodes_df
        .drop(columns='qvalue', errors='ignore')
        .merge(
            pathway_qvalues_df,
            on='id',
            how='left'
        )
    )

    # Store the DataFrame of quadripartite network edges
    file_name = NETWORK_FILES['quadripartite-edges']
    lqn_edges_df.to_csv(
        os.path.join(group_dir_path, file_name), index=False
    )

    # Store the DataFrame of quadripartite network nodes
    file_name = NETWORK_FILES['quadripartite-nodes']
    lqn_nodes_df.to_csv(
        os.path.join(group_dir_path, file_name), index=False
    )

    return lqn_edges_df, lqn_nodes_df

# Groups

## Basal-like

In [5]:
# Build the group's quadripartite network
group = 'Basal-like'
edges_df, nodes_df = build_quadripartite_network(group)

In [6]:
# Print the group's edges DataFrame
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-miR-23a-3p,RXRG,NaN,-0.467,0.007
1,PL09093,R-HSA-383280,NaN,NaN,NaN
2,PL09094,R-HSA-383280,NaN,NaN,NaN
3,RXRG,PL09093,NaN,NaN,NaN
4,RXRG,PL09094,NaN,NaN,NaN


In [7]:
# Print the group's nodes DataFrame
nodes_df

,id,label,type,qvalue
0,hsa-miR-23a-3p,hsa-miR-23a-3p,MicroRNA,NaN
1,RXRG,RXRG,Messenger RNA,NaN
2,R-HSA-383280,Nuclear Receptor transcription pathway,Pathway,0.064488
3,PL09093,Formation of NR-NCOR Corepressor Complex,Pathway Location,NaN
4,PL09094,Formation of NR-MED1 Coactivator Complex,Pathway Location,NaN


## HER2-enrichment

In [8]:
# Build the group's quadripartite network
group = 'HER2-enriched'
edges_df, nodes_df = build_quadripartite_network(group)

In [9]:
# Print the group's edges DataFrame
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-let-7d-5p,MEF2D,MIRT735654,-0.562,0.032
1,hsa-miR-106b-5p,SALL1,NaN,-0.545,0.042
2,hsa-let-7d-5p,PRKAR2A,NaN,-0.537,0.042
3,hsa-miR-155-5p,VAV3,NaN,-0.537,0.042
4,PL16682,R-HSA-1266738,NaN,NaN,NaN
5,PL16692,R-HSA-1266738,NaN,NaN,NaN
6,PL16950,R-HSA-1266738,NaN,NaN,NaN
7,PL17209,R-HSA-1266738,NaN,NaN,NaN
8,PL17498,R-HSA-1266738,NaN,NaN,NaN
9,PL17500,R-HSA-1266738,NaN,NaN,NaN


In [10]:
# Print the group's nodes DataFrame
nodes_df

,id,label,type,qvalue
0,hsa-let-7d-5p,hsa-let-7d-5p,MicroRNA,NaN
1,hsa-miR-106b-5p,hsa-miR-106b-5p,MicroRNA,NaN
2,hsa-miR-155-5p,hsa-miR-155-5p,MicroRNA,NaN
3,MEF2D,MEF2D,Messenger RNA,NaN
4,SALL1,SALL1,Messenger RNA,NaN
5,PRKAR2A,PRKAR2A,Messenger RNA,NaN
6,VAV3,VAV3,Messenger RNA,NaN
7,R-HSA-1266738,Developmental Biology,Pathway,0.075917
8,PL16682,Myogenesis > Phosphorylation of MEF2 proteins ...,Pathway Location,NaN
9,PL16692,Myogenesis > Interaction of MyoD:E protein wit...,Pathway Location,NaN


## Luminal A

In [11]:
# Build the group's quadripartite network
group = 'Luminal A'
edges_df, nodes_df = build_quadripartite_network(group)

In [12]:
# Print the group's edges DataFrame
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-miR-29c-3p,SERPINH1,MIRT437560,-0.419,0.0
1,hsa-miR-193b-3p,KIT,MIRT734593,-0.417,0.0
2,hsa-miR-141-3p,SOX17,NaN,-0.417,0.0
3,hsa-miR-210-3p,HIF3A,MIRT054794,-0.409,0.0
4,hsa-miR-96-5p,ADAM23,NaN,-0.384,0.0
...,...,...,...,...,...
338,NRP1,PL06862,NaN,NaN,NaN
339,NRP1,PL06863,NaN,NaN,NaN
340,NRP1,PL06864,NaN,NaN,NaN
341,SRGAP1,PL06894,NaN,NaN,NaN


In [13]:
# Print the group's nodes DataFrame
nodes_df

,id,label,type,qvalue
0,hsa-miR-200a-3p,hsa-miR-200a-3p,MicroRNA,NaN
1,hsa-miR-141-3p,hsa-miR-141-3p,MicroRNA,NaN
2,hsa-miR-96-5p,hsa-miR-96-5p,MicroRNA,NaN
3,hsa-miR-29b-3p,hsa-miR-29b-3p,MicroRNA,NaN
4,hsa-miR-342-3p,hsa-miR-342-3p,MicroRNA,NaN
...,...,...,...,...
197,PL06862,Axon guidance > Signaling by ROBO receptors > ...,Pathway Location,NaN
198,PL06863,Axon guidance > L1CAM interactions > Signal tr...,Pathway Location,NaN
199,PL06864,Axon guidance > L1CAM interactions > CHL1 inte...,Pathway Location,NaN
200,PL06894,Axon guidance > Signaling by ROBO receptors > ...,Pathway Location,NaN


## Luminal B

In [14]:
# Build the group's quadripartite network
group = 'Luminal B'
edges_df, nodes_df = build_quadripartite_network(group)

In [15]:
# Print the group's edges DataFrame
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-miR-30d-5p,FST,NaN,-0.461,0.001
1,hsa-miR-27b-3p,NEDD4,NaN,-0.448,0.001
2,hsa-miR-381-3p,ESR1,NaN,-0.428,0.002
3,hsa-miR-200c-3p,DZIP1,NaN,-0.397,0.004
4,hsa-miR-342-3p,PDGFRA,NaN,-0.378,0.006
...,...,...,...,...,...
491,PL03754,R-HSA-5663202,NaN,NaN,NaN
492,PL03755,R-HSA-5663202,NaN,NaN,NaN
493,TGFBR2,PL03753,NaN,NaN,NaN
494,TGFBR2,PL03754,NaN,NaN,NaN


In [16]:
# Print the group's nodes DataFrame
nodes_df

,id,label,type,qvalue
0,hsa-miR-200c-3p,hsa-miR-200c-3p,MicroRNA,NaN
1,hsa-miR-30d-5p,hsa-miR-30d-5p,MicroRNA,NaN
2,hsa-miR-27b-3p,hsa-miR-27b-3p,MicroRNA,NaN
3,hsa-miR-381-3p,hsa-miR-381-3p,MicroRNA,NaN
4,hsa-miR-33a-5p,hsa-miR-33a-5p,MicroRNA,NaN
...,...,...,...,...
273,PL13780,Interferon Signaling > Antiviral mechanism by ...,Pathway Location,NaN
274,R-HSA-5663202,Diseases of signal transduction by growth fact...,Pathway,0.095842
275,PL03753,Signaling by TGF-beta Receptor Complex in Canc...,Pathway Location,NaN
276,PL03754,Signaling by TGF-beta Receptor Complex in Canc...,Pathway Location,NaN


## Normal

In [17]:
# Build the group's quadripartite network
group = 'Normal'
edges_df, nodes_df = build_quadripartite_network(group)

In [18]:
# Print the group's edges DataFrame
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-miR-22-3p,RPS3,NaN,-0.816,0.0
1,hsa-miR-452-5p,RPA1,NaN,-0.810,0.0
2,hsa-miR-22-3p,RPL14,NaN,-0.804,0.0
3,hsa-miR-22-3p,LRRC1,MIRT030660,-0.800,0.0
4,hsa-miR-141-3p,FZD4,NaN,-0.799,0.0
...,...,...,...,...,...
6043,PL03018,R-HSA-8853884,NaN,NaN,NaN
6044,TP53,PL03014,NaN,NaN,NaN
6045,ANAPC15,PL03017,NaN,NaN,NaN
6046,PL04720,R-HSA-425381,NaN,NaN,NaN


In [19]:
# Print the group's nodes DataFrame
nodes_df

,id,label,type,qvalue
0,hsa-miR-22-3p,hsa-miR-22-3p,MicroRNA,NaN
1,hsa-miR-452-5p,hsa-miR-452-5p,MicroRNA,NaN
2,hsa-miR-141-3p,hsa-miR-141-3p,MicroRNA,NaN
3,hsa-miR-200a-3p,hsa-miR-200a-3p,MicroRNA,NaN
4,hsa-miR-224-5p,hsa-miR-224-5p,MicroRNA,NaN
...,...,...,...,...
2985,R-HSA-8853884,Transcriptional Regulation by VENTX,Pathway,0.093989
2986,PL03014,VENTX stimulates TP53 gene expression [ACTIVAT...,Pathway Location,NaN
2987,PL03017,Regulation of IL6 transcription [ACTIVATION],Pathway Location,NaN
2988,R-HSA-425381,Bicarbonate transporters,Pathway,0.095304
